In [1]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 3s (3,751 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 121703 files and dire

# 데이터 불러오기

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon
import tspoon as tsp

In [3]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [4]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [5]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [6]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

/content/gdrive/MyDrive/GDP/데이터


# GDP

In [7]:
gdp = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','110602').reset_index()
gdp

,index,숙박 및 음식점업
0,2014Q1,11088.1
1,2014Q2,11063.2
2,2014Q3,11327.1
3,2014Q4,11258.6
4,2015Q1,11239.4
5,2015Q2,11062.4
6,2015Q3,11150.4
7,2015Q4,11444.7
8,2016Q1,11643.7
9,2016Q2,11718.3


# PPI

In [8]:
ppi = getECOS('404Y014','Q','2014Q1','2025Q2','502AA').reset_index()
ppi

,index,음식점및숙박서비스
0,2014Q1,86.80
1,2014Q2,87.31
2,2014Q3,88.02
3,2014Q4,88.12
4,2015Q1,88.70
5,2015Q2,89.22
6,2015Q3,89.69
7,2015Q4,90.06
8,2016Q1,90.69
9,2016Q2,91.25


In [9]:
ppi = ppi.rename(columns={'음식점및숙박서비스':'ppi'})

# BSI(단순평균으로 계산)

In [10]:
업황실적BSI = getECOS('512Y007','M','201401','202506','AA','I5500').reset_index()
매출실적BSI = getECOS('512Y007','M','201401','202506','AB','I5500').reset_index()
채산성실적BSI = getECOS('512Y007','M','201401','202506','AE','I5500').reset_index()
자금사정실적BSI = getECOS('512Y007','M','201401','202506','AO','I5500').reset_index()
인력사정실적BSI = getECOS('512Y007','M','201401','202506','AJ','I5500').reset_index()

업황실적BSI = 업황실적BSI.rename(columns={'업황실적BSI 1)':'업황실적BSI'})
매출실적BSI = 매출실적BSI.rename(columns={'매출실적BSI 2)':'매출실적BSI'})
채산성실적BSI = 채산성실적BSI.rename(columns={'채산성실적BSI 6)':'채산성실적BSI'})
자금사정실적BSI = 자금사정실적BSI.rename(columns={'자금사정실적BSI 6)':'자금사정실적BSI'})
인력사정실적BSI = 인력사정실적BSI.rename(columns={'인력사정실적BSI 3)':'인력사정실적BSI'})


In [11]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI
0,201401,59.0,63.0,70.0,76.0,85.0
1,201402,76.0,74.0,80.0,83.0,83.0
2,201403,64.0,68.0,91.0,86.0,77.0
3,201404,80.0,93.0,85.0,96.0,87.0
4,201405,67.0,78.0,71.0,73.0,91.0
...,...,...,...,...,...,...
133,202502,49.0,58.0,53.0,64.0,61.0
134,202503,48.0,57.0,68.0,72.0,62.0
135,202504,46.0,63.0,68.0,70.0,60.0
136,202505,72.0,79.0,72.0,79.0,62.0


In [12]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())

    index    업황실적BSI    매출실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI      연도     월
0  2014Q1  66.333333  68.333333  80.333333  81.666667  81.666667  2014.0   2.0
1  2014Q2  69.333333  81.333333  76.333333  81.333333  88.000000  2014.0   5.0
2  2014Q3  74.333333  70.000000  80.333333  81.000000  85.000000  2014.0   8.0
3  2014Q4  80.000000  84.333333  94.666667  92.000000  78.000000  2014.0  11.0
4  2015Q1  62.000000  78.666667  79.000000  72.666667  84.000000  2015.0   2.0


In [13]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')

In [14]:
dfs = [ppi, gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,숙박 및 음식점업
0,2014Q1,66.333333,68.333333,80.333333,81.666667,81.666667,86.80,11088.1
1,2014Q2,69.333333,81.333333,76.333333,81.333333,88.000000,87.31,11063.2
2,2014Q3,74.333333,70.000000,80.333333,81.000000,85.000000,88.02,11327.1
3,2014Q4,80.000000,84.333333,94.666667,92.000000,78.000000,88.12,11258.6
4,2015Q1,62.000000,78.666667,79.000000,72.666667,84.000000,88.70,11239.4
5,2015Q2,56.333333,64.666667,69.000000,70.333333,84.666667,89.22,11062.4
6,2015Q3,49.333333,46.666667,66.666667,71.333333,87.000000,89.69,11150.4
7,2015Q4,61.333333,63.666667,79.333333,75.333333,81.666667,90.06,11444.7
8,2016Q1,57.333333,69.333333,80.333333,75.000000,76.666667,90.69,11643.7
9,2016Q2,68.333333,95.666667,95.666667,90.666667,76.000000,91.25,11718.3


In [15]:
df_final.to_csv("숙박음식점1차.csv")

# 생산지수

In [16]:
생산지수 = getKOSIS('DT_1F02001','Q','201401','202502','T20','101','00','C28')

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,94.449,2014Q1
1,97.056,2014Q2
2,93.755,2014Q3
3,96.695,2014Q4
4,90.577,2015Q1
5,90.360,2015Q2
6,93.073,2015Q3
7,94.494,2015Q4
8,95.813,2016Q1
9,92.740,2016Q2


내수/수출출하지수. 생산능력지수&가동률지수는 빼기~~

# 기업 데이터

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
df = pd.read_csv('임지오기업.csv')

In [19]:
숙박및음식점 = df[df['매핑한 산업'] == "숙박 및 음식점업"]

In [20]:
columns = ['EBITDA', '인건비']
grouped = 숙박및음식점.groupby('조사일')[columns].sum()
grouped

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,420068.0
2000-04-01,0.000000e+00,1540792.0
2000-07-01,0.000000e+00,1116188.0
2000-10-01,1.199656e+10,12018705.0
2001-01-01,0.000000e+00,994883.0
...,...,...
2024-04-01,4.729392e+10,29459084.0
2024-07-01,6.608675e+10,28831887.0
2024-10-01,5.084804e+10,157731010.0


In [21]:
grouped = grouped.reset_index()

In [22]:
grouped['조사일']=pd.to_datetime(grouped['조사일'])

# 분기(Quarter) 단위로 변환
grouped["index"] = grouped["조사일"].dt.to_period("Q")
grouped = grouped.drop(columns=["조사일"])
print(grouped)

grouped

           EBITDA          인건비   index
0    0.000000e+00     420068.0  2000Q1
1    0.000000e+00    1540792.0  2000Q2
2    0.000000e+00    1116188.0  2000Q3
3    1.199656e+10   12018705.0  2000Q4
4    0.000000e+00     994883.0  2001Q1
..            ...          ...     ...
97   4.729392e+10   29459084.0  2024Q2
98   6.608675e+10   28831887.0  2024Q3
99   5.084804e+10  157731010.0  2024Q4
100  4.961662e+10   19360356.0  2025Q1
101  2.493236e+10  -19360356.0  2025Q2

[102 rows x 3 columns]


,EBITDA,인건비,index
0,0.000000e+00,420068.0,2000Q1
1,0.000000e+00,1540792.0,2000Q2
2,0.000000e+00,1116188.0,2000Q3
3,1.199656e+10,12018705.0,2000Q4
4,0.000000e+00,994883.0,2001Q1
...,...,...,...
97,4.729392e+10,29459084.0,2024Q2
98,6.608675e+10,28831887.0,2024Q3
99,5.084804e+10,157731010.0,2024Q4
100,4.961662e+10,19360356.0,2025Q1


In [23]:
mask = (grouped["index"] >= pd.Period("2014Q1")) & (grouped["index"] <= pd.Period("2025Q2"))
기업= grouped.loc[mask]
기업
기업['합산'] = 기업['EBITDA']+기업['인건비']
기업

/tmp/ipython-input-1389489221.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업['합산'] = 기업['EBITDA']+기업['인건비']


,EBITDA,인건비,index,합산
56,6.743750e+09,7236861.0,2014Q1,6.750987e+09
57,1.600951e+10,6737337.0,2014Q2,1.601624e+10
58,1.681186e+10,5751428.0,2014Q3,1.681761e+10
59,4.794518e+10,66649305.0,2014Q4,4.801183e+10
60,1.904878e+10,8217871.0,2015Q1,1.905700e+10
61,2.794872e+10,7973961.0,2015Q2,2.795670e+10
62,2.450258e+10,8752023.0,2015Q3,2.451133e+10
63,6.111635e+10,140937472.0,2015Q4,6.125728e+10
64,1.158239e+10,10573264.0,2016Q1,1.159296e+10
65,4.512304e+10,19654395.0,2016Q2,4.514269e+10


# 전체 데이터 합산

In [24]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [df_final,생산지수, 기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [df_final,생산지수, 기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)

/tmp/ipython-input-994347461.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['index'] = df['index'].astype(str)


In [25]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [26]:
df_merged

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,숙박 및 음식점업,생산지수,EBITDA,인건비,합산
0,2014Q1,66.333333,68.333333,80.333333,81.666667,81.666667,86.80,11088.1,94.449,6.743750e+09,7236861.0,6.750987e+09
1,2014Q2,69.333333,81.333333,76.333333,81.333333,88.000000,87.31,11063.2,97.056,1.600951e+10,6737337.0,1.601624e+10
2,2014Q3,74.333333,70.000000,80.333333,81.000000,85.000000,88.02,11327.1,93.755,1.681186e+10,5751428.0,1.681761e+10
3,2014Q4,80.000000,84.333333,94.666667,92.000000,78.000000,88.12,11258.6,96.695,4.794518e+10,66649305.0,4.801183e+10
4,2015Q1,62.000000,78.666667,79.000000,72.666667,84.000000,88.70,11239.4,90.577,1.904878e+10,8217871.0,1.905700e+10
5,2015Q2,56.333333,64.666667,69.000000,70.333333,84.666667,89.22,11062.4,90.360,2.794872e+10,7973961.0,2.795670e+10
6,2015Q3,49.333333,46.666667,66.666667,71.333333,87.000000,89.69,11150.4,93.073,2.450258e+10,8752023.0,2.451133e+10
7,2015Q4,61.333333,63.666667,79.333333,75.333333,81.666667,90.06,11444.7,94.494,6.111635e+10,140937472.0,6.125728e+10
8,2016Q1,57.333333,69.333333,80.333333,75.000000,76.666667,90.69,11643.7,95.813,1.158239e+10,10573264.0,1.159296e+10
9,2016Q2,68.333333,95.666667,95.666667,90.666667,76.000000,91.25,11718.3,92.740,4.512304e+10,19654395.0,4.514269e+10


In [27]:
df_merged.to_csv('숙박및음식점.csv')

# 데이터QoQ반환

In [28]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = df_merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())

    index    업황실적BSI    매출실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1        NaN        NaN        NaN        NaN        NaN       NaN   
1  2014Q2   4.522613  19.024390  -4.979253  -0.408163   7.755102  0.587558   
2  2014Q3   7.211538 -13.934426   5.240175  -0.409836  -3.409091  0.813194   
3  2014Q4   7.623318  20.476190  17.842324  13.580247  -8.235294  0.113611   
4  2015Q1 -22.500000  -6.719368 -16.549296 -21.014493   7.692308  0.658193   

   숙박 및 음식점업      생산지수      EBITDA          인건비          합산  
0        NaN       NaN         NaN          NaN         NaN  
1  -0.224565  2.760220  137.397679    -6.902495  137.242994  
2   2.385386 -3.401129    5.011729   -14.633512    5.003465  
3  -0.604744  3.135833  185.186671  1058.830555  185.485447  
4  -0.170536 -6.327111  -60.269663   -87.669982  -60.307700  


qoq 타겟은 gdp 숙박 및 음식

In [29]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])

In [30]:
df_qoq.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
채산성실적BSI,0
자금사정실적BSI,0
인력사정실적BSI,0
ppi,0
숙박 및 음식점업,0
생산지수,0
EBITDA,0


In [31]:
df_qoq.to_csv('숙박및음식점qoq.csv')

# 데이터YoY변환

In [32]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = df_merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))

    index    업황실적BSI    매출실적BSI   채산성실적BSI  자금사정실적BSI  인력사정실적BSI       ppi  \
0  2014Q1        NaN        NaN        NaN        NaN        NaN       NaN   
1  2014Q2        NaN        NaN        NaN        NaN        NaN       NaN   
2  2014Q3        NaN        NaN        NaN        NaN        NaN       NaN   
3  2014Q4        NaN        NaN        NaN        NaN        NaN       NaN   
4  2015Q1  -6.532663  15.121951  -1.659751 -11.020408   2.857143  2.188940   
5  2015Q2 -18.750000 -20.491803  -9.606987 -13.524590  -3.787879  2.187607   
6  2015Q3 -33.632287 -33.333333 -17.012448 -11.934156   2.352941  1.897296   
7  2015Q4 -23.333333 -24.505929 -16.197183 -18.115942   4.700855  2.201543   
8  2016Q1  -7.526882 -11.864407   1.687764   3.211009  -8.730159  2.243517   
9  2016Q2  21.301775  47.938144  38.647343  28.909953 -10.236220  2.275275   

   숙박 및 음식점업      생산지수      EBITDA         인건비          합산  
0        NaN       NaN         NaN         NaN         NaN  
1        NaN       

In [33]:
df_yoy=df_yoy.dropna()

In [34]:
df_yoy

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,숙박 및 음식점업,생산지수,EBITDA,인건비,합산
4,2015Q1,-6.532663,15.121951,-1.659751,-11.020408,2.857143,2.188940,1.364526,-4.099567,182.465720,13.555739,182.284654
5,2015Q2,-18.750000,-20.491803,-9.606987,-13.524590,-3.787879,2.187607,-0.007231,-6.899110,74.575811,18.354789,74.552162
6,2015Q3,-33.632287,-33.333333,-17.012448,-11.934156,2.352941,1.897296,-1.559976,-0.727428,45.745786,52.171304,45.747983
7,2015Q4,-23.333333,-24.505929,-16.197183,-18.115942,4.700855,2.201543,1.652959,-2.276229,27.471301,111.461278,27.587895
8,2016Q1,-7.526882,-11.864407,1.687764,3.211009,-8.730159,2.243517,3.597167,5.780717,-39.196181,28.661840,-39.166918
9,2016Q2,21.301775,47.938144,38.647343,28.909953,-10.236220,2.275275,5.929093,2.633909,61.449365,146.482206,61.473618
10,2016Q3,47.972973,157.142857,40.000000,27.102804,-17.624521,2.252202,5.702038,1.369892,26.344675,-29.677139,26.324672
11,2016Q4,14.673913,57.068063,15.966387,21.238938,-4.489796,2.043082,1.714331,2.949394,-44.871943,14.559706,-44.735206
12,2017Q1,0.000000,9.615385,-7.053942,-4.000000,5.217391,2.205315,0.064413,1.203386,174.799202,35.819904,174.672447
13,2017Q2,2.439024,-12.891986,-16.027875,-11.397059,1.315789,2.257534,0.438630,6.389907,-46.688551,-27.024297,-46.679989


In [35]:
df_yoy.to_csv("숙박및음식점yoy.csv")

# 상관계수 보기(qoq)

In [36]:
qoq = pd.read_csv('숙박및음식점qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = '숙박 및 음식점업'

In [38]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)

📊 GDP 대비 단순 상관계수
숙박 및 음식점업    1.000000
자금사정실적BSI    0.484753
채산성실적BSI     0.409879
매출실적BSI      0.401198
업황실적BSI      0.363287
인력사정실적BSI    0.178586
ppi          0.165251
합산           0.121180
EBITDA       0.121108
인건비          0.074717
생산지수         0.029312
Name: 숙박 및 음식점업, dtype: float64


In [39]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = '숙박 및 음식점업'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df

/tmp/ipython-input-836008556.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  corr_df = pd.concat(


,Feature,Lag,Correlation
0,매출실적BSI,2,0.473928
1,자금사정실적BSI,2,0.407207
2,업황실적BSI,2,0.366603
3,채산성실적BSI,2,0.365513
4,ppi,1,0.300980
5,인력사정실적BSI,4,0.298291
6,생산지수,2,0.262468
7,생산지수,3,0.239447
8,매출실적BSI,4,0.200948
9,ppi,2,0.135039


In [40]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,숙박 및 음식점업,생산지수,EBITDA,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
0,2015Q1,-22.500000,-6.719368,-16.549296,-21.014493,7.692308,0.658193,-0.170536,-6.327111,-60.269663,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015Q2,-9.139785,-17.796610,-12.658228,-3.211009,0.793651,0.586246,-1.574817,-0.239575,46.721848,...,NaN,NaN,-87.669982,NaN,NaN,NaN,-60.307700,NaN,NaN,NaN
2,2015Q3,-12.426036,-27.835052,-3.381643,1.421801,2.755906,0.526788,0.795487,3.002435,-12.330255,...,NaN,NaN,-2.968044,-87.669982,NaN,NaN,46.700421,-60.307700,NaN,NaN
3,2015Q4,24.324324,36.428571,19.000000,5.607477,-6.130268,0.412532,2.639367,1.526759,149.428248,...,-60.269663,NaN,9.757535,-2.968044,-87.669982,NaN,-12.323955,46.700421,-60.307700,NaN
4,2016Q1,-6.521739,8.900524,1.260504,-0.442478,-6.122449,0.699534,1.738796,1.395856,-81.048627,...,46.721848,-60.269663,1510.341655,9.757535,-2.968044,-87.669982,149.914176,-12.323955,46.700421,-60.307700
5,2016Q2,19.186047,37.980769,19.087137,20.888889,-0.869565,0.617488,0.640690,-3.207289,289.583244,...,-12.330255,46.721848,-92.497904,1510.341655,9.757535,-2.968044,-81.074968,149.914176,-12.323955,46.700421
6,2016Q3,6.829268,25.435540,-2.439024,0.000000,-5.701754,0.504110,0.579436,1.733880,-31.392697,...,149.428248,-12.330255,85.887679,-92.497904,1510.341655,9.757535,289.397465,-81.074968,149.914176,-12.323955
7,2016Q4,-3.652968,-16.666667,-1.428571,0.735294,8.837209,0.207175,-1.232798,3.108704,8.833195,...,-81.048627,149.428248,-68.685513,85.887679,-92.497904,1510.341655,-31.408934,289.397465,-81.074968,149.914176
8,2017Q1,-18.483412,-24.000000,-18.840580,-21.167883,3.418803,0.859630,0.088481,-0.323804,-5.532272,...,289.583244,-81.048627,2523.332775,-68.685513,85.887679,-92.497904,9.333001,-31.408934,289.397465,-81.074968
9,2017Q2,22.093023,9.649123,7.589286,11.574074,-4.545455,0.668896,1.017063,1.753192,-24.420278,...,-31.392697,289.583244,-91.105652,2523.332775,-68.685513,85.887679,-5.940394,9.333001,-31.408934,289.397465


In [41]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

target_corr = corr["숙박 및 음식점업"].drop("숙박 및 음식점업")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,숙박 및 음식점업
자금사정실적BSI,0.484753
매출실적BSI_lag2,0.473928
채산성실적BSI,0.409879
자금사정실적BSI_lag2,0.407207
매출실적BSI,0.401198
업황실적BSI_lag2,0.366603
채산성실적BSI_lag2,0.365513
업황실적BSI,0.363287
ppi_lag1,0.300980
인력사정실적BSI_lag4,0.298291


In [42]:
target_corr.to_csv('숙박및음식점순서.csv')

In [43]:
df2.dropna(inplace=True)

In [44]:
df2

,index,업황실적BSI,매출실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,ppi,숙박 및 음식점업,생산지수,EBITDA,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
4,2016Q1,-6.521739,8.900524,1.260504,-0.442478,-6.122449,0.699534,1.738796,1.395856,-81.048627,...,46.721848,-60.269663,1510.341655,9.757535,-2.968044,-87.669982,149.914176,-12.323955,46.700421,-60.307700
5,2016Q2,19.186047,37.980769,19.087137,20.888889,-0.869565,0.617488,0.640690,-3.207289,289.583244,...,-12.330255,46.721848,-92.497904,1510.341655,9.757535,-2.968044,-81.074968,149.914176,-12.323955,46.700421
6,2016Q3,6.829268,25.435540,-2.439024,0.000000,-5.701754,0.504110,0.579436,1.733880,-31.392697,...,149.428248,-12.330255,85.887679,-92.497904,1510.341655,9.757535,289.397465,-81.074968,149.914176,-12.323955
7,2016Q4,-3.652968,-16.666667,-1.428571,0.735294,8.837209,0.207175,-1.232798,3.108704,8.833195,...,-81.048627,149.428248,-68.685513,85.887679,-92.497904,1510.341655,-31.408934,289.397465,-81.074968,149.914176
8,2017Q1,-18.483412,-24.000000,-18.840580,-21.167883,3.418803,0.859630,0.088481,-0.323804,-5.532272,...,289.583244,-81.048627,2523.332775,-68.685513,85.887679,-92.497904,9.333001,-31.408934,289.397465,-81.074968
9,2017Q2,22.093023,9.649123,7.589286,11.574074,-4.545455,0.668896,1.017063,1.753192,-24.420278,...,-31.392697,289.583244,-91.105652,2523.332775,-68.685513,85.887679,-5.940394,9.333001,-31.408934,289.397465
10,2017Q3,-10.952381,-10.000000,1.244813,2.904564,-7.792208,0.675169,1.380664,-0.288853,61.766020,...,8.833195,-31.392697,-0.123003,-91.105652,2523.332775,-68.685513,-24.409320,-5.940394,9.333001,-31.408934
11,2017Q4,11.229947,0.000000,5.737705,-2.016129,-2.347418,0.276772,-2.010526,2.035962,-52.202894,...,-5.532272,8.833195,93.492286,-0.123003,-91.105652,2523.332775,61.784926,-24.409320,-5.940394,9.333001
12,2018Q1,-12.500000,-0.888889,-12.015504,-7.407407,13.461538,1.146497,3.745200,-3.514504,133.383938,...,-24.420278,-5.532272,505.590497,93.492286,-0.123003,-91.105652,-51.805373,61.784926,-24.409320,-5.940394
13,2018Q2,24.175824,21.076233,25.991189,17.333333,-5.508475,0.997061,0.075019,1.885273,-14.180231,...,61.766020,-24.420278,-88.353853,505.590497,93.492286,-0.123003,131.398273,-51.805373,61.784926,-24.409320


In [45]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])

In [46]:
df2.to_csv('숙박및음식점lag.csv', encoding='utf-8-sig')

# ARIMA

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('숙박및음식점lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '숙박 및 음식점업'   # GDP 예측 대상 칼럼
#상관계수 보고 넣기
exog_vars = ['자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2','매출실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (3.0, 1.0, 0.0)
📉 평균 Train MAE: 2.8832
📈 평균 Test MAE: 5.5283


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('숙박및음식점lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '숙박 및 음식점업'   # GDP 예측 대상 칼럼
#상관계수 보고 넣기
exog_vars = ['자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2', '인력사정실적BSI_lag3']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 3.33
📈 평균 Test MAE: 4.7945


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('숙박및음식점lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '숙박 및 음식점업'   # GDP 예측 대상 칼럼
#상관계수 보고 넣기
exog_vars = ['자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 5)
d_values = range(1, 5)
q_values = range(0, 5)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 3.3156
📈 평균 Test MAE: 4.2984


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('숙박및음식점lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '숙박 및 음식점업'   # GDP 예측 대상 칼럼
#상관계수 보고 넣기
exog_vars = ['자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (0.0, 1.0, 1.0)
📉 평균 Train MAE: 3.4673
📈 평균 Test MAE: 4.5825


## 아리마 예측값

In [47]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('숙박및음식점lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = ['자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2']

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    pred = fit.forecast(steps=1, exog=X_test)

                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 (p,d,q) 선택
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

print("✅ 최적 ARIMAX(p,d,q):", (best_p, best_d, best_q))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# ---------------------------------------------------
# 8️⃣ 최적 (p,d,q)로 전체 구간 롤링 예측값 저장
# ---------------------------------------------------
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(train_data) < train_window or len(X_test) == 0:
        continue

    try:
        model = SARIMAX(train_data, exog=X_train, order=(best_p, best_d, best_q),
                        enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        pred = fit.forecast(steps=1, exog=X_test)

        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y.loc[test_end])
        })
    except:
        continue

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 최적 모델 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 ARIMAX(p,d,q): (0, 1, 1)
📉 평균 Train MAE: 3.3156
📈 평균 Test MAE: 4.2984

📌 최적 모델 예측값(pred) 시계열:
             pred     actual
date                        
2021Q1   5.107692   3.896117
2021Q2   0.847124   8.666941
2021Q3  -0.006125   2.380776
2021Q4  18.531040  10.380341
2022Q1  -2.899599  -8.770666
2022Q2   5.154966  12.004773
2022Q3  -1.469302   5.125556
2022Q4   1.863603   0.659078
2023Q1  -3.123893  -3.542529
2023Q2   0.023554  -6.829686
2023Q3  -4.327943   2.357529
2023Q4  -1.897137   2.494960
2024Q1  -1.493127   1.021219
2024Q2  -2.544207  -6.654269
2024Q3  -0.320361   1.472041
2024Q4  -2.613691   2.788103
2025Q1  -1.470550  -1.509289
2025Q2   0.408112  -4.666871


In [48]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 12115.142414614113


# 랜덤포레스트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = ['업황실적BSI', '매출실적BSI', '채산성실적BSI', '자금사정실적BSI', '인력사정실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.7828
📈 평균 Test MAE: 4.9432


## 랜덤포레스트 예측값

In [49]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = ['자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간 1회)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

rf = RandomForestRegressor(random_state=42)
rand_search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = RandomForestRegressor(**best_params, random_state=42)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 정리
# -----------------------------
print("\n📊 랜덤포레스트 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 랜덤포레스트 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'n_estimators': 200, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': 12}

📊 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.4323
📈 평균 Test MAE: 4.7907

📌 랜덤포레스트 예측값(pred) 시계열:
            pred     actual
date                       
2021Q1 -0.311505   3.896117
2021Q2 -0.517205   8.666941
2021Q3  1.077980   2.380776
2021Q4  1.743146  10.380341
2022Q1  1.321331  -8.770666
2022Q2  4.693030  12.004773
2022Q3  0.781927   5.125556
2022Q4  1.476159   0.659078
2023Q1  0.611401  -3.542529
2023Q2  0.803412  -6.829686
2023Q3 -5.177980   2.357529
2023Q4 -0.579205   2.494960
2024Q1 -0.126123   1.021219
2024Q2 -2.317878  -6.654269
2024Q3 -0.954306   1.472041
2024Q4  0.819918   2.788103
2025Q1 -0.245646  -1.509289
2025Q2  2.131608  -4.666871


In [50]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 12323.097685345825


# AR

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점qoq.csv')

# GDP 단일 시계열만 사용 (AR(1))
y = df['숙박 및 음식점업'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 3.9133
📈 평균 Test MAE: 5.9664


## AR 예측값

In [51]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점qoq.csv')

# GDP 단일 시계열
y = df['숙박 및 음식점업'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# ⭐ 예측값 저장할 리스트
rolling_preds = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train MAE ---
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(test_data.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

# -----------------------------
# 5️⃣ pred 시계열 데이터프레임 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 AR(1) 예측값(pred) 시계열:")
print(pred_df)


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 3.9133
📈 평균 Test MAE: 5.9664

📌 AR(1) 예측값(pred) 시계열:
             pred     actual
date                        
2020Q1   0.304321 -13.281005
2020Q2  10.104973  -2.940088
2020Q3  -0.240277  -3.225327
2020Q4  -0.629891  -7.997233
2021Q1  -1.890255   3.896117
2021Q2  -0.924036   8.666941
2021Q3   0.686645   2.380776
2021Q4   0.255835  10.380341
2022Q1   2.677845  -8.770666
2022Q2  -0.027848  12.004773
2022Q3  -1.928083   5.125556
2022Q4   0.440536   0.659078
2023Q1   0.651272  -3.542529
2023Q2   0.719210  -6.829686
2023Q3   0.228775   2.357529
2023Q4   0.056823   2.494960
2024Q1   0.017793   1.021219
2024Q2   0.186311  -6.654269
2024Q3   0.145308   1.472041
2024Q4  -0.221709   2.788103
2025Q1   0.679893  -1.509289
2025Q2   0.981444  -4.666871


In [52]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 12184.320038169086


# XGB

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = ['업황실적BSI', '매출실적BSI', '채산성실적BSI', '자금사정실적BSI', '인력사정실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 1.0}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.0011
📈 평균 Test MAE: 5.9841


## XGB 예측값

In [53]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = [
'자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2']

y = df[target].dropna()
X = df[exog_vars].dropna()

df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ RandomizedSearchCV (초기 구간)
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []   # ⭐ 예측값 저장

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 XGBoost 예측값(pred) 시계열:")
print(pred_df)


✅ 최적 하이퍼파라미터: {'subsample': 1.0, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.1763
📈 평균 Test MAE: 4.3470

📌 XGBoost 예측값(pred) 시계열:
            pred     actual
date                       
2021Q1 -0.806515   3.896117
2021Q2 -0.170143   8.666941
2021Q3 -0.300291   2.380776
2021Q4  3.207093  10.380341
2022Q1  0.168124  -8.770666
2022Q2  8.417093  12.004773
2022Q3  1.267620   5.125556
2022Q4 -0.851689   0.659078
2023Q1 -2.083658  -3.542529
2023Q2 -0.563058  -6.829686
2023Q3 -8.418320   2.357529
2023Q4  3.596442   2.494960
2024Q1 -8.557642   1.021219
2024Q2 -3.641119  -6.654269
2024Q3  0.517254   1.472041
2024Q4  2.291292   2.788103
2025Q1  0.011286  -1.509289
2025Q2 -6.456302  -4.666871


In [54]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 11286.889094489097


# MLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = ['업황실적BSI', '매출실적BSI', '채산성실적BSI', '자금사정실적BSI', '인력사정실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'lbfgs', 'learning_rate_init': 0.01, 'hidden_layer_sizes': (64, 32), 'alpha': 0.01, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.0228
📈 평균 Test MAE: 5.6790


##MLP 예측값

In [55]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = [
'자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2']

df_model = df[[target] + exog_vars].dropna()

y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링 (정상 코드)
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 5️⃣ 하이퍼파라미터 축소 (안정성)
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16)],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.005]
}

first_train_end = start_period + (train_window - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=2000)
rand_search = RandomizedSearchCV(
    mlp, param_grid, n_iter=5, cv=3,
    scoring='neg_mean_absolute_error', n_jobs=-1, random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 파라미터:", best_params)

# -----------------------------
# 6️⃣ 롤링 예측
# -----------------------------
train_mae_list = []
test_mae_list = []
rolling_preds = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    X_train = X_scaled.loc[train_start:train_end]
    y_train = y_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]
    y_test = y_scaled.loc[test_end:test_end]

    if len(X_train) < train_window:
        continue

    try:
        model = MLPRegressor(
            **best_params,
            random_state=42,
            max_iter=2000
        )
        model.fit(X_train, y_train)

        # ---- Train 예측 ----
        train_pred_scaled = model.predict(X_train)
        train_pred = scaler_y.inverse_transform(train_pred_scaled.reshape(-1,1)).flatten()
        train_real = scaler_y.inverse_transform(y_train.values.reshape(-1,1)).flatten()
        train_mae = mean_absolute_error(train_real, train_pred)
        train_mae_list.append(train_mae)

        # ---- Test 예측 ----
        test_pred_scaled = model.predict(X_test)
        test_pred = scaler_y.inverse_transform(test_pred_scaled.reshape(-1,1)).flatten()
        test_real = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()
        test_mae = mean_absolute_error(test_real, test_pred)
        test_mae_list.append(test_mae)

        # ---- pred 저장 ----
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred[0]),
            'actual': float(test_real[0])
        })

    except Exception as e:
        print("❌ 오류 발생:", e)
        continue

# -----------------------------
# 7️⃣ 결과 출력
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")

if len(rolling_preds) > 0:
    pred_df = pd.DataFrame(rolling_preds).set_index('date')
    print(pred_df)
else:
    print("⚠ rolling_preds가 텅 빔 → 모델 학습 실패")


✅ 최적 파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (64,), 'alpha': 0.001, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.9049
📈 평균 Test MAE: 4.9863
            pred     actual
date                       
2021Q1  3.102399   3.896117
2021Q2 -2.812739   8.666941
2021Q3 -6.482350   2.380776
2021Q4  1.962025  10.380341
2022Q1  0.090185  -8.770666
2022Q2  1.671373  12.004773
2022Q3  0.215552   5.125556
2022Q4  0.296375   0.659078
2023Q1  3.945826  -3.542529
2023Q2 -0.858827  -6.829686
2023Q3 -4.242568   2.357529
2023Q4 -2.233016   2.494960
2024Q1  2.243583   1.021219
2024Q2 -2.091571  -6.654269
2024Q3 -0.171666   1.472041
2024Q4 -0.269073   2.788103
2025Q1 -1.473965  -1.509289
2025Q2 -4.243613  -4.666871


In [57]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 11553.869878619169


# LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = ['업황실적BSI', '매출실적BSI', '채산성실적BSI', '자금사정실적BSI', '인력사정실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 2.0939
📈 평균 Test MAE: 5.1000


## LSTM 예측값

In [58]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ y, X 설정
# -----------------------------
target = '숙박 및 음식점업'
exog_vars = [
'자금사정실적BSI','매출실적BSI_lag2','채산성실적BSI','자금사정실적BSI_lag2'
]

df_model = pd.concat([df[target], df[exog_vars]], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(
    scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(),
    index=y.index
)

# -----------------------------
# 4️⃣ LSTM 입력 생성
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ 모델 생성 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측 + pred 저장
# -----------------------------
train_mae_list, test_mae_list = [], []
rolling_preds = []  # ⭐ 예측값 저장 리스트

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    train_start = train_end - (train_window - 1)
    test_end = test_start

    # LSTM용 슬라이싱
    idx_arr = np.array(idxs)

    train_mask = (idx_arr >= train_start) & (idx_arr <= train_end)
    test_mask  = (idx_arr == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test,  y_test  = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

        model.fit(X_train, y_train, epochs=200, batch_size=8,
                  verbose=0, callbacks=[early])

        # ---- Train 예측 ----
        train_pred = model.predict(X_train, verbose=0)
        test_pred  = model.predict(X_test, verbose=0)

        # ---- 스케일 복원 ----
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1,1))

        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_test_inv  = scaler_y.inverse_transform(y_test.values.reshape(-1,1))

        # ---- MAE ----
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(test_pred_inv[0]),
            'actual': float(y_test_inv[0])
        })

    except Exception as e:
        print(f"❌ Error: {e}")
        continue

# -----------------------------
# 8️⃣ 결과 요약 + pred_df 출력
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE:  {np.mean(test_mae_list):.4f}")

# ⭐ 예측값 시계열 출력
pred_df = pd.DataFrame(rolling_preds).set_index('date')
print("\n📌 LSTM 예측값(pred) 시계열:")
print(pred_df)



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 2.7979
📈 평균 Test MAE:  5.4552

📌 LSTM 예측값(pred) 시계열:
             pred     actual
date                        
2022Q1   4.734954  -8.770666
2022Q2  -4.770379  12.004773
2022Q3   1.688489   5.125556
2022Q4  10.395676   0.659078
2023Q1   0.152952  -3.542529
2023Q2  -2.481303  -6.829686
2023Q3  -0.657938   2.357529
2023Q4  -0.495886   2.494960
2024Q1   3.202907   1.021219
2024Q2   0.966346  -6.654269
2024Q3   0.796187   1.472041
2024Q4   1.721965   2.788103
2025Q1   1.331302  -1.509289
2025Q2  -0.183992  -4.666871


In [59]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 12043.699677097678


# 단변량 아리마

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

# 인덱스가 분기형일 경우 변환
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상 설정
# -----------------------------
y = df['숙박 및 음식점업'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간
start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 탐색 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                # 데이터 슬라이싱
                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    # -------------------------
                    # 모델 학습
                    # -------------------------
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    # -------------------------
                    # 예측 및 MAE 계산
                    # -------------------------
                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception as e:
                    print(f"❌ Error at (p,d,q)=({p},{d},{q}): {e}")
                    continue

            # -------------------------
            # 평균 MAE 저장
            # -------------------------
            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 결과가 없습니다. 모든 (p,d,q) 조합이 실패했습니다.")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    print("✅ 최적 ARIMA(p,d,q):", tuple(best[['p', 'd', 'q']]))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

    display(results_df.sort_values('test_MAE').head(10))


✅ 최적 ARIMA(p,d,q): (0.0, 1.0, 2.0)
📉 평균 Train MAE: 4.4207
📈 평균 Test MAE: 5.6982


,p,d,q,train_MAE,test_MAE
2,0,1,2,4.420720,5.698176
1,0,1,1,4.476214,5.712667
38,3,1,2,4.206993,5.908254
37,3,1,1,4.392962,6.093932
13,1,1,1,4.487718,6.270936
36,3,1,0,4.488517,6.307713
25,2,1,1,4.414994,6.525884
14,1,1,2,4.494908,6.602809
24,2,1,0,4.510241,6.610607
27,2,1,3,4.079028,6.657395


## 단변량 아리마 예측값

In [60]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('숙박및음식점lag.csv')

df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상
# -----------------------------
y = df['숙박 및 음식점업'].dropna()

# -----------------------------
# 3️⃣ 롤링 윈도우
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2021Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ (p,d,q) 범위
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                y_train = y.loc[train_start:train_end]
                y_test = y.loc[test_end:test_end]

                if len(y_train) < train_window:
                    continue

                try:
                    model = ARIMA(y_train, order=(p, d, q))
                    fit = model.fit()

                    pred = fit.forecast(steps=1)

                    train_mae = mean_absolute_error(y_train, fit.fittedvalues)
                    test_mae = mean_absolute_error(y_test, pred)

                    train_mae_list.append(train_mae)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 최적 모델 선택
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    print("⚠️ 모든 조합 실패")
else:
    best = results_df.loc[results_df['test_MAE'].idxmin()]
    best_p, best_d, best_q = int(best['p']), int(best['d']), int(best['q'])

    print("✅ 최적 ARIMA(p,d,q):", (best_p, best_d, best_q))
    print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
    print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

# -----------------------------
# 8️⃣ ⭐ 최적 모델로 pred 저장 (재예측)
# -----------------------------
rolling_preds = []
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]

    if len(y_train) < train_window:
        continue

    try:
        model = ARIMA(y_train, order=(best_p, best_d, best_q))
        fit = model.fit()

        pred = fit.forecast(steps=1)

        # MAE
        train_mae_list.append(mean_absolute_error(y_train, fit.fittedvalues))
        test_mae_list.append(mean_absolute_error(y_test, pred))

        # ⭐ 예측값 저장
        rolling_preds.append({
            'date': test_end,
            'pred': float(pred),
            'actual': float(y_test.values[0])
        })

    except Exception as e:
        print("❌ 재예측 오류:", e)
        continue

# -----------------------------
# 9️⃣ pred_df 출력
# -----------------------------
pred_df = pd.DataFrame(rolling_preds).set_index('date')

print("\n📌 ARIMA 최적모델 예측(pred) 시계열:")
print(pred_df)

print("\n📉 최종 Train MAE:", np.mean(train_mae_list))
print("📈 최종 Test MAE:", np.mean(test_mae_list))


✅ 최적 ARIMA(p,d,q): (0, 1, 2)
📉 평균 Train MAE: 4.4207
📈 평균 Test MAE: 5.6982

📌 ARIMA 최적모델 예측(pred) 시계열:
             pred     actual
date                        
2021Q1  -5.912892   3.896117
2021Q2  -4.715916   8.666941
2021Q3   7.992007   2.380776
2021Q4  11.003500  10.380341
2022Q1   6.237073  -8.770666
2022Q2  -0.393445  12.004773
2022Q3   2.479864   5.125556
2022Q4   4.580264   0.659078
2023Q1   4.418183  -3.542529
2023Q2   0.618706  -6.829686
2023Q3  -0.061921   2.357529
2023Q4   0.189647   2.494960
2024Q1   0.295806   1.021219
2024Q2   0.210489  -6.654269
2024Q3  -0.208777   1.472041
2024Q4  -0.031387   2.788103
2025Q1   0.088182  -1.509289
2025Q2   0.679382  -4.666871

📉 최종 Train MAE: 4.42072011359544
📈 최종 Test MAE: 5.698175553946481


In [61]:
level_2025Q1 = 12065.9

qoq_2025Q2_pred = pred_df.loc['2025Q2', 'pred']
level_2025Q2_pred = level_2025Q1 * (1 + qoq_2025Q2_pred / 100)

print("2025Q2 수준값(예측):", level_2025Q2_pred)

2025Q2 수준값(예측): 12147.873550207038
